# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asadnaeem23/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
import duckdb

token = userdata.get("HF")
print("HF token loaded:", bool(token))

con = duckdb.connect()

con.execute(
    f"CREATE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{token}')"
)

rel = "hf://datasets/FlyRank/internship-warehouse"

print("DuckDB connected.")

HF token loaded: True
DuckDB connected.


In [2]:
path = f"{rel}/fact_content_daily_performance/**/*.parquet"

con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{path}', hive_partitioning=true)
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


### Unit of analysis and time window

One row represents one content item for one client on one report date.

The table used is `fact_content_daily_performance`. The data is recorded at a daily level and is partitioned by month.

For this notebook, I will use March 2026 (`2026-03`) as the development and verification window. The final month is not used for this development work.

The decision-support task is to use information observed up to the decision date to rank existing content items by refresh priority. Future observations are not used as features.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify the unit of analysis and the March 2026 time window.

query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id) AS distinct_grain_keys,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet',
    hive_partitioning=true
)
WHERE month = '2026-03'
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_keys,start_date,end_date
0,9841378,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields

#### Features

The initial feature set contains five measured fields:

1. `gsc_impressions` — measured Google Search Console impressions available for the observation.
2. `gsc_clicks` — measured Google Search Console clicks available for the observation.
3. `gsc_avg_position` — measured average search position available for the observation.
4. `ga4_sessions` — measured GA4 sessions available for the observation.
5. `scroll_events` — measured engagement events available for the observation.

These are candidate features because they describe observed search performance and user engagement.

#### Label / proxy

There is no directly observed refresh-success label in `fact_content_daily_performance`.

The task therefore uses a refresh-priority proxy for decision-support purposes rather than claiming to predict the causal effect of refreshing content.

#### Context

- `report_date` — identifies when the observation was measured.
- `client_hash_id` — identifies the anonymized client.
- `content_hash_id` — identifies the anonymized content item.
- `client_has_gsc` — indicates whether the client has GSC data.
- `client_has_ga4` — indicates whether the client has GA4 data.
- `gsc_data_available` — indicates whether GSC data is available for the observation.
- `ga4_data_available` — indicates whether GA4 data is available for the observation.
- `month` — identifies the warehouse partition month.

#### Excluded

- `report_date` is excluded as a model feature because it identifies time rather than content performance.
- `client_hash_id` and `content_hash_id` are excluded because they are identifiers, not generalizable performance signals.
- `month` is excluded because it is a partition/context field.
- Data-availability and client-capability flags are kept as context for this first contract rather than treating availability itself as content performance.
- Future observations are excluded because using information from after the decision moment would create leakage.
- No label-derived field is included as a feature because it would leak the outcome into the model.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify that the five proposed feature fields exist in the warehouse.

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events"
]

schema_check = con.sql(f"""
    SELECT column_name, column_type
    FROM (
        DESCRIBE
        SELECT *
        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet',
            hive_partitioning=true
        )
    )
    WHERE column_name IN ({",".join("'" + c + "'" for c in feature_columns)})
    ORDER BY column_name
""").df()

schema_check

,column_name,column_type
0,ga4_sessions,BIGINT
1,gsc_avg_position,DOUBLE
2,gsc_clicks,BIGINT
3,gsc_impressions,BIGINT
4,scroll_events,BIGINT


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Verify the data contract for March 2026.
# Query 1 verifies the grain.
# Query 2 verifies the row count and date window.
# Query 3 verifies data availability and missingness.

march_path = f"{rel}/fact_content_daily_performance/**/*.parquet"

# Query 1: Grain
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id)
            AS distinct_grain_keys
    FROM read_parquet(
        '{march_path}',
        hive_partitioning=true
    )
    WHERE month = '2026-03'
""").df()

print("Query 1 — Grain")
display(grain_check)


# Query 2: Row count and date window
window_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM read_parquet(
        '{march_path}',
        hive_partitioning=true
    )
    WHERE month = '2026-03'
""").df()

print("Query 2 — March 2026 window")
display(window_check)


# Query 3: Data availability
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)
            AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)
            AS ga4_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE)
            AS gsc_unavailable_or_missing_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE)
            AS ga4_unavailable_or_missing_rows
    FROM read_parquet(
        '{march_path}',
        hive_partitioning=true
    )
    WHERE month = '2026-03'
""").df()

print("Query 3 — Data availability")
display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 — Grain


,total_rows,distinct_grain_keys
0,9841378,9841378


Query 2 — March 2026 window


,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 3 — Data availability


,total_rows,gsc_available_rows,ga4_available_rows,gsc_unavailable_or_missing_rows,ga4_unavailable_or_missing_rows
0,9841378,3611061,413966,6230317,9427412



### Leakage trap

A leakage check was performed by intentionally creating a feature from the decision target/proxy.

The leaked feature contains information derived from the outcome that would not be available at the decision moment. Such a feature can make a model appear artificially strong because it gives the model direct information about what it is supposed to predict.

This is not an acceptable production feature. After demonstrating the leakage risk, the label-derived feature is removed and the honest feature set contains only information available at the decision moment.

The leakage example is used only as a diagnostic and is not retained in the final feature set.

In [6]:
# Leakage demonstration
# This is intentionally leaked and must NOT be used as a real feature.

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Small March sample for the demonstration
leak_df = con.sql(f"""
    SELECT
        gsc_impressions,
        gsc_clicks
    FROM read_parquet(
        '{march_path}',
        hive_partitioning=true
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND gsc_impressions > 0
      AND gsc_clicks >= 0
    LIMIT 10000
""").df()

# Define a simple demonstration proxy.
# This is only for showing the leakage mechanism.
leak_df["proxy_label"] = (leak_df["gsc_clicks"] > 0).astype(int)

# INTENTIONAL LEAK:
# The feature directly contains the label.
leak_df["leaked_feature"] = leak_df["proxy_label"]

X = leak_df[["leaked_feature"]]
y = leak_df["proxy_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

model = DecisionTreeClassifier(max_depth=2, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

leaked_accuracy = accuracy_score(y_test, predictions)

print(f"Accuracy with intentional leakage: {leaked_accuracy:.4f}")

# Remove the leaked feature.
leak_df = leak_df.drop(columns=["leaked_feature"])

print("Leaked feature removed.")
print("Final feature set does not contain label-derived information.")

Accuracy with intentional leakage: 1.0000
Leaked feature removed.
Final feature set does not contain label-derived information.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This data contains measured daily performance observations, but it cannot by itself tell us whether refreshing a piece of content caused its future performance to improve.

The history is not balanced across all data sources. In March 2026, GSC and GA4 availability differs across observations, so the absence of a metric may reflect unavailable source data rather than zero activity.

The data also contains daily observations, so multiple rows can belong to the same content item across different dates. This means historical windows can overlap when creating features from multiple dates.

The warehouse provides observed performance signals such as impressions, clicks, sessions, and engagement, but it does not provide a directly observed refresh-success outcome. Therefore, refresh priority should be treated as directional decision-support rather than a causal prediction.

The final month should remain sealed from feature development and model decisions. Future observations must not be used to construct features for an earlier decision date.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.